# Clase 07 — Ejercicio guiado: tu primera red neuronal en Keras
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

En clase vimos **qué** hace una red neuronal: capas de unidades, una no linealidad entre ellas,
una función de costo y descenso por el gradiente. En este notebook la vas a **construir y
entrenar tú**, con **Keras**, sobre un problema pequeño que puedes ver dibujado.

Keras es la API de alto nivel de TensorFlow. Su gracia es que toda una red neuronal se reduce a
cuatro verbos, y ese es exactamente el esqueleto de este notebook:

| Verbo | Qué hace | Qué concepto de clase encapsula |
|---|---|---|
| **construir** (`Sequential`) | apilar capas | la arquitectura: capas, unidades y activaciones |
| **compilar** (`compile`) | elegir cómo se entrena | la función de costo $\mathcal{L}$ y el optimizador |
| **entrenar** (`fit`) | ajustar los pesos | *forward* → costo → **backpropagation** → actualización |
| **evaluar** (`evaluate`, `predict`) | medir sobre datos no vistos | generalización |

> **Lo que Keras te oculta —y conviene no olvidar—**: `fit()` ejecuta, para cada mini-batch,
> exactamente el ciclo que vimos en las diapositivas: propagación hacia adelante, cálculo del
> error, propagación del error hacia atrás por la regla de la cadena, y actualización de los
> pesos. No es magia; es el mismo algoritmo, escrito en una sola línea.

---

## Cómo trabajar este notebook

- Las celdas marcadas con `# TODO` son **tuyas**: están incompletas a propósito.
- Después de cada `TODO` hay una **celda de validación** con `assert`. Si imprime `OK`, puedes
  seguir; si falla, el mensaje te dice qué revisar.
- Los **experimentos** de la sección 8 traen una pregunta antes de cada celda. Escribe tu
  predicción **antes** de ejecutar: equivocarte prediciendo es la forma más rápida de entender
  qué hace cada hiperparámetro.
- Al final hay **preguntas de cierre** para responder en las celdas de texto.

> **Tiempo de ejecución**: el notebook completo tarda unos 4–5 minutos en CPU. Los experimentos
> de la sección 8 entrenan varias redes seguidas y son la parte más lenta.

---

---
## 0. Configuración del entorno

Necesitamos **NumPy** (números), **Matplotlib** (gráficas), **scikit-learn** (solo para generar y
partir el dataset) y **Keras** (la red).

Fijamos una **semilla aleatoria** con `keras.utils.set_random_seed`, que siembra a la vez a
Python, NumPy y TensorFlow. Sin ella, la inicialización aleatoria de los pesos haría que cada
ejecución diera un resultado distinto y no podríamos comparar experimentos.

In [ ]:
import os
# Silencia los mensajes informativos de TensorFlow. Debe ir ANTES de importarlo.
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import keras

SEED = 42
keras.utils.set_random_seed(SEED)   # siembra Python + NumPy + TensorFlow de una sola vez

print(f'Keras   : {keras.__version__}')
print(f'Backend : {keras.backend.backend()}')
print(f'Semilla : {SEED}')

---
## 1. El dataset *toy*: "dos lunas"

Vamos a usar `make_moons`: dos grupos de puntos en forma de media luna entrelazadas.

¿Por qué este dataset y no Fashion-MNIST, como en el notebook de ejemplo?

1. **Vive en 2 dimensiones**, así que podemos **dibujar la frontera de decisión** y *ver*
   exactamente lo que la red aprendió. Con 784 features eso es imposible.
2. **No es linealmente separable**: ninguna recta separa las dos lunas. Es el mismo obstáculo que
   frenó al perceptrón, y la razón por la que necesitamos capas ocultas.
3. Es pequeño: cada entrenamiento toma segundos, así que puedes experimentar muchas veces.

Es, de hecho, uno de los datasets de [playground.tensorflow.org](http://playground.tensorflow.org).

### Notación (la de la clase)

- $\mathbf{X} \in \Re^{n \times d}$: matriz de características, $n$ ejemplos y $d=2$ features.
- $y^{(i)}$ vale $0$ o $1$: a qué luna pertenece el punto $i$. Es un problema **binario**.

In [ ]:
# 1200 puntos en dos medias lunas entrelazadas.
# noise=0.20 añade ruido gaussiano: sin él las lunas serían perfectas y el problema trivial.
X, y = make_moons(n_samples=1200, noise=0.20, random_state=SEED)
y = y.astype('float32')          # Keras espera float en la etiqueta para pérdida binaria

print(f'Shape de X : {X.shape}   -> (n_ejemplos, d_features)')
print(f'Shape de y : {y.shape}')
print(f'Clases     : {np.unique(y).astype(int)}  |  ejemplos por clase: '
      f'{int((y==0).sum())} / {int((y==1).sum())}')
print(f'\nPrimeras 3 filas de X:\n{X[:3].round(3)}')
print(f'Sus etiquetas: {y[:3].astype(int)}')

In [ ]:
# Visualizamos el dataset completo: cada punto es un ejemplo, el color es su clase.
plt.figure(figsize=(6, 5))
plt.scatter(X[y == 0, 0], X[y == 0, 1], c='#c0392b', edgecolors='k',
            linewidths=0.4, s=25, label='Clase 0')
plt.scatter(X[y == 1, 0], X[y == 1, 1], c='#2471a3', edgecolors='k',
            linewidths=0.4, s=25, label='Clase 1')
plt.xlabel('$x_1$'); plt.ylabel('$x_2$')
plt.title('Dataset "dos lunas" (make_moons)')
plt.legend(); plt.grid(alpha=0.2)
plt.show()

> **Míralo antes de seguir.** Intenta trazar mentalmente **una sola recta** que deje todos los
> puntos rojos de un lado y todos los azules del otro. No existe. Por eso necesitamos una capa
> oculta con una activación no lineal — y en la sección 8 vas a comprobar en vivo qué pasa si se
> la quitas.

### Partición en tres conjuntos y estandarización

Igual que en el notebook de Fashion-MNIST, partimos en **tres** conjuntos:

| Conjunto | Para qué sirve |
|---|---|
| **Entrenamiento** (~66%) | ajustar los pesos con backpropagation |
| **Validación** (~17%) | vigilar el modelo **durante** el entrenamiento y decidir hiperparámetros |
| **Prueba** (~17%) | medir el desempeño **final**, una sola vez, sobre datos nunca vistos |

> **¿Por qué no basta con dos?** Si usáramos el conjunto de prueba para elegir el número de
> neuronas o el *learning rate*, lo estaríamos optimizando indirectamente y su accuracy dejaría de
> ser una estimación honesta. Para eso está el de validación.

Además **estandarizamos**: transformamos cada feature a media $0$ y desviación $1$. El descenso
por el gradiente converge mucho mejor cuando todas las features están en la misma escala.

> **Detalle metodológico**: $\mu$ y $\sigma$ se calculan **solo** con el conjunto de entrenamiento
> y se aplican a los otros dos. Calcularlos sobre el dataset completo sería *data leakage*.

In [ ]:
# Primero separamos entrenamiento del resto; luego partimos el resto en validación y prueba.
# stratify mantiene la proporción de clases en cada subconjunto.
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.34, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED
)

# Estandarización: mu y sigma salen SOLO del conjunto de entrenamiento.
mu    = X_train.mean(axis=0)
sigma = X_train.std(axis=0)

# float32 es el tipo por defecto de Keras: evita conversiones silenciosas.
X_train = ((X_train - mu) / sigma).astype('float32')
X_val   = ((X_val   - mu) / sigma).astype('float32')
X_test  = ((X_test  - mu) / sigma).astype('float32')

print(f'Entrenamiento : {X_train.shape[0]:>5} ejemplos')
print(f'Validación    : {X_val.shape[0]:>5} ejemplos')
print(f'Prueba        : {X_test.shape[0]:>5} ejemplos')
print(f'\nMedia por columna (train)      : {X_train.mean(axis=0).round(6)}  -> ~0')
print(f'Desviación por columna (train) : {X_train.std(axis=0).round(6)}  -> ~1')
print(f'Media por columna (test)       : {X_test.mean(axis=0).round(3)}   <- NO es exactamente 0, y está bien')

---
## 2. Construir la red

La arquitectura que vamos a implementar es la más pequeña que resuelve el problema:

```
   x (2 features)  ──Dense(8, relu)──>  capa oculta  ──Dense(1, sigmoid)──>  P(y=1|x)
```

Cada `Dense` es una **capa fully-connected**: aplica $\mathbf{W}\mathbf{x} + \mathbf{b}$ y luego
la activación. Es literalmente la unidad de la clase, repetida tantas veces como neuronas pidas.

### Del concepto de clase a la línea de Keras

| En la clase | En Keras |
|---|---|
| Una neurona: $g(\mathbf{w}\cdot\mathbf{x} + b)$ | una unidad dentro de un `Dense` |
| Capa oculta de $H$ unidades con activación $g$ | `keras.layers.Dense(H, activation='relu')` |
| Salida con sigmoide → probabilidad | `keras.layers.Dense(1, activation='sigmoid')` |
| Apilar capas | `keras.Sequential([...])` |

### Por qué la salida es `Dense(1, activation='sigmoid')`

El problema es **binario**, así que basta **una** neurona de salida: su valor es
$P(y=1\mid\mathbf{x})$, y $P(y=0\mid\mathbf{x})$ es el complemento. La sigmoide es la que aplasta
cualquier número real al intervalo $(0,1)$ para que se pueda leer como probabilidad.

> **Comparación con Fashion-MNIST**: allá había 10 clases, así que la salida era
> `Dense(10, activation='softmax')`. Softmax es la generalización de la sigmoide a $K$ clases.
> El resto de la red no cambia en nada.

**TODO 1.** Construye el modelo. Necesitas tres piezas dentro de la lista:

1. `keras.Input(shape=(2,))` — declara que cada ejemplo tiene 2 features. No es una capa con
   pesos: solo le dice a Keras la forma de la entrada para que pueda construir la red.
2. La capa oculta: `keras.layers.Dense(8, activation='relu')`.
3. La capa de salida: `keras.layers.Dense(1, activation='sigmoid')`.

In [ ]:
# TODO 1: completa la lista de capas.
# Recuerda el orden: Input -> capa oculta -> capa de salida.
model = keras.Sequential([
    ...,   # keras.Input con shape=(2,)
    ...,   # capa oculta: 8 unidades, activación relu
    ...,   # capa de salida: 1 unidad, activación sigmoid
])

In [ ]:
# --- Validación TODO 1 ---
assert isinstance(model, keras.Sequential), 'model debe ser un keras.Sequential'
assert len(model.layers) == 2, (f'Deben quedar 2 capas con pesos (la oculta y la de salida), '
                                f'hay {len(model.layers)}. Ojo: keras.Input NO cuenta como capa.')
assert model.layers[0].units == 8, f'La capa oculta debe tener 8 unidades, tiene {model.layers[0].units}'
assert model.layers[0].activation.__name__ == 'relu', \
    f"La capa oculta debe usar relu, usa '{model.layers[0].activation.__name__}'"
assert model.layers[-1].units == 1, 'La capa de salida debe tener 1 unidad (problema binario)'
assert model.layers[-1].activation.__name__ == 'sigmoid', \
    f"La salida debe usar sigmoid, usa '{model.layers[-1].activation.__name__}'"
assert model.output_shape == (None, 1), f'La salida debe ser (None, 1), es {model.output_shape}'
assert model.count_params() == 33, f'Deberían salir 33 parámetros, salieron {model.count_params()}'

print('OK TODO 1 - arquitectura correcta.\n')
model.summary()

### Lee el `summary()`: ¿de dónde salen los 33 parámetros?

Cada capa `Dense` tiene una matriz de pesos y un vector de sesgos:

$$\underbrace{2 \times 8}_{\mathbf{W}^{[1]}} + \underbrace{8}_{\mathbf{b}^{[1]}} = 24
\qquad
\underbrace{8 \times 1}_{\mathbf{W}^{[2]}} + \underbrace{1}_{\mathbf{b}^{[2]}} = 9
\qquad
24 + 9 = 33$$

Treinta y tres números: eso es *toda* la red. Entrenarla es encontrar los 33 valores que
minimizan el costo.

> **Un chequeo que vale la pena adquirir como hábito**: si `summary()` no te da el número de
> parámetros que esperabas, la arquitectura no es la que creías. Es el error más común y el más
> fácil de detectar.

---
## 3. Compilar: elegir el costo y el optimizador

`compile()` no entrena nada. Solo configura **cómo** se va a entrenar, y son las dos casillas que
en clase acompañan siempre al modelo:

| Casilla de la clase | Argumento de `compile` | Aquí |
|---|---|---|
| función de costo $\mathcal{L}$ | `loss=` | `'binary_crossentropy'` |
| procedimiento de optimización | `optimizer=` | `keras.optimizers.Adam(learning_rate=0.01)` |
| — (solo para mirar) | `metrics=` | `['accuracy']` |

### La pérdida: entropía cruzada binaria

Es la función que vimos en clase, para el caso de dos clases:

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\Big[y^{(i)}\log \hat{y}^{(i)} + (1-y^{(i)})\log(1-\hat{y}^{(i)})\Big]$$

Castiga sin límite la confianza equivocada. Cuando el modelo no sabe nada y predice $0.5$ para
todo, esta pérdida vale $\log 2 \approx 0.69$ — **memoriza ese número**: es tu línea base para
saber si el entrenamiento está haciendo algo.

> Con 10 clases usaríamos `'sparse_categorical_crossentropy'`, como en Fashion-MNIST. Es la
> misma idea con softmax en lugar de sigmoide.

### El optimizador: Adam

**Adam** es descenso por el gradiente con dos mejoras: acumula inercia (*momentum*) y adapta el
tamaño del paso para cada parámetro por separado. Sigue siendo, en esencia, el
$\theta \leftarrow \theta - \alpha\,\partial\mathcal{L}/\partial\theta$ de la Parte 1 de la clase.

**TODO 2.** Compila el modelo con esos tres argumentos y `learning_rate=0.01`.

In [ ]:
# TODO 2: completa los tres argumentos de compile().
model.compile(
    optimizer=...,   # Adam con learning_rate=0.01
    loss=...,        # entropía cruzada binaria
    metrics=...,     # lista con 'accuracy'
)

In [ ]:
# --- Validación TODO 2 ---
assert model.optimizer is not None, 'El modelo no quedó compilado'
assert type(model.optimizer).__name__ == 'Adam', \
    f'Se esperaba el optimizador Adam, llegó {type(model.optimizer).__name__}'
assert abs(float(model.optimizer.learning_rate) - 0.01) < 1e-6, \
    f'learning_rate debería ser 0.01, es {float(model.optimizer.learning_rate)}'
assert 'binary' in str(model.loss).lower(), \
    f'La pérdida debe ser entropía cruzada BINARIA, llegó: {model.loss}'

print('OK TODO 2 - modelo compilado.')
print(f'  optimizador   : {type(model.optimizer).__name__} (lr = {float(model.optimizer.learning_rate)})')
print(f'  función costo : {model.loss}')
print(f'\nPérdida de referencia (modelo que no sabe nada): log(2) = {np.log(2):.4f}')

---
## 4. Entrenar

`model.fit()` ejecuta el ciclo completo. Para **cada mini-batch** de cada época hace exactamente
lo que vimos en las diapositivas de backpropagation:

```
forward  →  calcular la pérdida  →  propagar el error hacia atrás  →  actualizar los pesos
```

### Los argumentos que importan

- **`batch_size=32`**: cuántos ejemplos se procesan antes de cada actualización de pesos. Con 791
  ejemplos de entrenamiento salen $\lceil 791/32 \rceil = 25$ actualizaciones por época.
- **`epochs=200`**: cuántas veces se recorre el dataset completo.
- **`validation_data=(X_val, y_val)`**: al final de cada época, Keras evalúa sobre validación.
  Esos datos **no** se usan para actualizar pesos — solo para mirar.
- **`verbose=2`**: una línea por época. (`0` = silencio, `1` = barra de progreso.)

**TODO 3.** Llama a `model.fit` con esos argumentos y guarda el resultado en `history`.

> Tarda unos 20 segundos. Mientras corre, fíjate en `loss`: debe bajar desde ~0.69.

In [ ]:
# TODO 3: entrena el modelo.
# Guarda lo que devuelve fit() en la variable `history`: contiene el historial de métricas.
history = ...

In [ ]:
# --- Validación TODO 3 ---
h = history.history        # dict con el historial: 'loss', 'accuracy', 'val_loss', 'val_accuracy'

assert set(h.keys()) >= {'loss', 'accuracy', 'val_loss', 'val_accuracy'}, \
    'Falta validation_data en fit(): sin él no hay val_loss ni val_accuracy'
assert len(h['loss']) == 200, f"Deberían ser 200 épocas, hay {len(h['loss'])}"
assert h['loss'][-1] < h['loss'][0], 'La pérdida debería BAJAR durante el entrenamiento'
assert h['loss'][-1] < 0.30, (f"La pérdida final es {h['loss'][-1]:.3f}: demasiado alta. "
                              f"Si quedó cerca de 0.69 el modelo no aprendió nada.")
assert h['accuracy'][-1] > 0.93, f"Accuracy de entrenamiento baja: {h['accuracy'][-1]:.3f}"

print('OK TODO 3 - el entrenamiento funcionó.')
print(f"  pérdida  : {h['loss'][0]:.4f}  ->  {h['loss'][-1]:.4f}   (referencia log 2 = 0.6931)")
print(f"  accuracy : {h['accuracy'][0]:.4f}  ->  {h['accuracy'][-1]:.4f}")
print(f"  val_acc  : {h['val_accuracy'][0]:.4f}  ->  {h['val_accuracy'][-1]:.4f}")

---
## 5. Las curvas de aprendizaje

Son la primera herramienta de diagnóstico de cualquier entrenamiento. Qué mirar:

- La **pérdida de entrenamiento** debe bajar de forma sostenida. Si oscila o explota, el
  *learning rate* es demasiado grande; si apenas se mueve, demasiado pequeño.
- La **brecha** entre entrenamiento y validación mide el **sobreajuste**: si la de entrenamiento
  sigue bajando mientras la de validación empieza a subir, la red está memorizando. El punto donde
  eso empieza es donde conviene hacer *early stopping* (sección 8, experimento D).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(h['loss'], label='entrenamiento', lw=2)
axes[0].plot(h['val_loss'], label='validación', lw=2, ls='--')
axes[0].axhline(np.log(2), color='gray', ls=':', lw=1.5, label='"no saber nada" = log 2')
axes[0].set_xlabel('época'); axes[0].set_ylabel('entropía cruzada binaria')
axes[0].set_title('Curva de pérdida'); axes[0].legend(); axes[0].grid(alpha=0.2)

axes[1].plot(h['accuracy'], label='entrenamiento', lw=2)
axes[1].plot(h['val_accuracy'], label='validación', lw=2, ls='--')
axes[1].set_xlabel('época'); axes[1].set_ylabel('accuracy')
axes[1].set_title('Curva de accuracy'); axes[1].legend(); axes[1].grid(alpha=0.2)

plt.tight_layout(); plt.show()

print(f"Accuracy final - entrenamiento : {h['accuracy'][-1]:.4f}")
print(f"Accuracy final - validación    : {h['val_accuracy'][-1]:.4f}")

---
## 6. ¿Qué aprendió la red? La frontera de decisión

Aquí está la ventaja de trabajar en 2D: podemos evaluar el modelo en **todos** los puntos del
plano y colorear cada región según la probabilidad que predice. La línea negra es la frontera
$P(y=1\mid\mathbf{x}) = 0.5$: el lugar donde la red cambia de opinión.

La función siguiente **te la damos hecha**; úsala tal cual durante todo el notebook.

In [ ]:
def plot_decision_boundary(modelo, X, y, ax=None, title=''):
    """Dibuja la frontera de decisión de un modelo de Keras sobre datos 2D."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5.5, 4.5))

    # 1) Malla de puntos que cubre el rango de los datos, con un margen.
    x0 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200)
    x1 = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200)
    XX, YY = np.meshgrid(x0, x1)
    grid = np.c_[XX.ravel(), YY.ravel()].astype('float32')   # (40000, 2)

    # 2) Una sola llamada a predict para los 40.000 puntos de la malla.
    Z = modelo.predict(grid, verbose=0).reshape(XX.shape)

    # 3) Fondo = probabilidad;  línea negra = frontera P(y=1) = 0.5
    ax.contourf(XX, YY, Z, levels=np.linspace(0, 1, 21), cmap='RdBu', alpha=0.75)
    ax.contour(XX, YY, Z, levels=[0.5], colors='k', linewidths=1.6)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', edgecolors='k', s=22, linewidths=0.4)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$'); ax.set_title(title)
    return ax


plot_decision_boundary(model, X_test, y_test, title='Red 2-8-1 sobre el conjunto de prueba')
plt.show()

> **Mira la forma de la frontera.** Está hecha de **tramos rectos**: es la huella de ReLU, que es
> lineal a trozos. Cada una de las 8 neuronas ocultas aporta un "pliegue", y al combinarlos la red
> arma una frontera curva por partes. Con más neuronas habría más pliegues y se vería más suave.
>
> Esa es la intuición del **teorema de aproximación universal**: con suficientes neuronas puedes
> aproximar cualquier función continua tan bien como quieras.

---
## 7. Evaluación final

Hasta ahora solo hemos mirado entrenamiento y validación. El conjunto de **prueba** se toca
**una sola vez**, al final, y es el único número honesto sobre datos nunca vistos.

- `model.evaluate(X, y)` devuelve `[pérdida, accuracy]`.
- `model.predict(X)` devuelve las probabilidades $P(y=1\mid\mathbf{x})$, con shape `(n, 1)`.
  Para pasar de probabilidad a clase aplicamos el umbral $0.5$.

**TODO 4.** Evalúa sobre el conjunto de prueba y obtén las predicciones.

In [ ]:
# TODO 4a: evalúa el modelo sobre el conjunto de PRUEBA.
# evaluate devuelve una lista [loss, accuracy]; usa verbose=0 para que no imprima la barra.
test_loss, test_acc = ...

# TODO 4b: obtén las probabilidades predichas para X_test (shape (n, 1)).
proba_test = ...

# Convertimos probabilidades a clases con el umbral 0.5 (ya implementado).
pred_test = (proba_test.ravel() >= 0.5).astype('float32')

print(f'Pérdida en prueba  : {test_loss:.4f}')
print(f'Accuracy en prueba : {test_acc:.4f}  ({test_acc*100:.2f}% de {len(y_test)} ejemplos)')

In [ ]:
# --- Validación TODO 4 ---
assert 0.0 <= test_acc <= 1.0, 'test_acc debe ser una fracción entre 0 y 1'
assert test_acc > 0.90, f'El accuracy en prueba debería superar 0.90, es {test_acc:.3f}'
assert proba_test.shape == (len(y_test), 1), \
    f'predict debe devolver shape ({len(y_test)}, 1), devolvió {proba_test.shape}'
assert proba_test.min() >= 0 and proba_test.max() <= 1, \
    'predict debe devolver PROBABILIDADES en [0, 1] (la sigmoide ya está en el modelo)'
assert abs(float(np.mean(pred_test == y_test)) - test_acc) < 1e-4, \
    'El accuracy calculado a mano no coincide con el de evaluate: revisa el umbral'

print('OK TODO 4 - evaluación correcta.')
print(f'\nLas 5 primeras probabilidades: {proba_test[:5].ravel().round(3)}')
print(f'Sus clases predichas          : {pred_test[:5].astype(int)}')
print(f'Sus etiquetas reales          : {y_test[:5].astype(int)}')

### Matriz de confusión: dónde se equivoca

El accuracy es un solo número y esconde información. La matriz de confusión dice **qué** confunde
con **qué** — en el notebook de Fashion-MNIST era lo que revelaba que el modelo mezclaba
`camiseta` con `camisa`. Aquí, con dos clases, es una tabla de $2\times 2$.

In [ ]:
cm = confusion_matrix(y_test, pred_test)
VN, FP, FN, VP = cm.ravel()      # verdaderos neg., falsos pos., falsos neg., verdaderos pos.

print('Matriz de confusión (conjunto de prueba)')
print('                pred 0   pred 1')
print(f'  real 0      {VN:7d}  {FP:7d}')
print(f'  real 1      {FN:7d}  {VP:7d}')
print(f'\nAccuracy  : {(VP + VN) / len(y_test):.4f}')
print(f'Precisión : {VP / max(1, VP + FP):.4f}   (de lo que predije como 1, cuánto era 1)')
print(f'Recall    : {VP / max(1, VP + FN):.4f}   (de los 1 reales, cuántos encontré)')

# Marcamos los errores sobre el mapa de la frontera.
err = pred_test != y_test
fig, ax = plt.subplots(figsize=(6.5, 5))
plot_decision_boundary(model, X_test, y_test, ax, f'Errores en prueba ({int(err.sum())} de {len(y_test)})')
ax.scatter(X_test[err, 0], X_test[err, 1], marker='x', s=160, c='lime', linewidths=2.5)
plt.show()

print(f'\nConfianza de la red en los ejemplos que falló: {proba_test[err].ravel().round(3)}')
print('Casi todos caen pegados a la frontera, donde las dos lunas se solapan por el ruido.')

---
## 8. Experimentos guiados

Ahora la red es tu laboratorio. **Antes de ejecutar cada celda, escribe tu predicción.**

Para poder repetir entrenamientos cambiando una sola cosa, definimos una función que construye,
compila y entrena en un paso. Fíjate que vuelve a sembrar la semilla en cada llamada: así la única
diferencia entre dos experimentos es el hiperparámetro que cambiaste, no el azar.

In [ ]:
def entrenar(H=8, activation='relu', lr=0.01, epochs=200, capas_ocultas=1,
             datos=None, batch_size=32, callbacks=None, verbose=0):
    """Construye, compila y entrena una red 2 -> [H]*capas_ocultas -> 1.

    Devuelve (modelo, history). Vuelve a sembrar la semilla en cada llamada para que
    la unica diferencia entre dos experimentos sea el hiperparametro que cambiaste.
    """
    Xa, ya, Xb, yb = datos if datos is not None else (X_train, y_train, X_val, y_val)

    keras.utils.set_random_seed(SEED)          # misma inicialización en todos los experimentos

    capas = [keras.Input(shape=(2,))]
    for _ in range(capas_ocultas):
        capas.append(keras.layers.Dense(H, activation=activation))
    capas.append(keras.layers.Dense(1, activation='sigmoid'))

    m = keras.Sequential(capas)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
              loss='binary_crossentropy', metrics=['accuracy'])
    hh = m.fit(Xa, ya, batch_size=batch_size, epochs=epochs,
               validation_data=(Xb, yb), callbacks=callbacks, verbose=verbose)
    return m, hh


print('Función auxiliar lista. Cada llamada entrena una red desde cero.')

---
### Experimento A — Quitarle la no linealidad (el más importante)

Las diapositivas afirman: *sin activación no lineal, apilar capas no aporta nada, porque
$W_2(W_1x) = (W_2W_1)x = Wx$*. Vamos a comprobarlo.

Entrenamos **la misma red**: mismos 33 parámetros, misma inicialización, mismo optimizador.
Lo único que cambia es `activation=None` en la capa oculta.

**Predice primero**: ¿qué accuracy alcanzará? ¿Qué forma tendrá su frontera?

In [ ]:
model_lin, h_lin = entrenar(activation=None)     # activation=None -> capa puramente lineal

acc_lin = model_lin.evaluate(X_test, y_test, verbose=0)[1]
print(f'Red 2-8-1 CON ReLU       -> accuracy en prueba: {test_acc:.4f}')
print(f'Red 2-8-1 SIN activación -> accuracy en prueba: {acc_lin:.4f}')
print(f'Ambas tienen exactamente {model_lin.count_params()} parámetros.')

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
plot_decision_boundary(model_lin, X_test, y_test, axes[0], f'activation=None (acc={acc_lin:.3f})')
plot_decision_boundary(model,     X_test, y_test, axes[1], f'activation=relu (acc={test_acc:.3f})')
plt.tight_layout(); plt.show()

La frontera de la izquierda es una **recta**. No porque la red sea pequeña, sino porque
*no puede ser otra cosa*. Comprobémoslo con álgebra: si la capa oculta es lineal, toda la red
equivale a un único vector de pesos.

$$\sigma\big((\mathbf{X}\mathbf{W}^{[1]} + \mathbf{b}^{[1]})\mathbf{W}^{[2]} + \mathbf{b}^{[2]}\big)
= \sigma\big(\mathbf{X}\underbrace{\mathbf{W}^{[1]}\mathbf{W}^{[2]}}_{\text{un vector } (2,1)} + \text{constante}\big)$$

In [ ]:
# Extraemos los pesos aprendidos por la red "profunda" sin activación.
W1, b1 = model_lin.layers[0].get_weights()      # (2, 8) y (8,)
W2, b2 = model_lin.layers[1].get_weights()      # (8, 1) y (1,)

w_eq = W1 @ W2                 # (2, 1): los 24 pesos de la capa oculta colapsan en 2 números
b_eq = b1 @ W2 + b2            # (1,)

print(f'Pesos equivalentes : {w_eq.ravel().round(4)}')
print(f'Sesgo equivalente  : {b_eq.round(4)}')

# Comparamos la red de 33 parámetros contra el modelo lineal de 3 parámetros equivalente.
pred_red    = model_lin.predict(X_test, verbose=0).ravel()
pred_lineal = 1 / (1 + np.exp(-(X_test @ w_eq + b_eq).ravel()))

print(f'\nDiferencia máxima entre ambas predicciones: {np.max(np.abs(pred_red - pred_lineal)):.2e}')
print('Son el mismo modelo. Los 33 parámetros describen solo 3 grados de libertad:')
print('sin no linealidad, la profundidad es una ilusión.')

---
### Experimento B — ¿Cuántas neuronas ocultas hacen falta?

$H$ controla la **capacidad**: cuántos pliegues puede tener la frontera.

**Predice**: ¿qué pasa con $H=1$? ¿Mejora siempre al aumentar $H$?

> Tarda alrededor de un minuto y medio: son cinco entrenamientos.

In [ ]:
resultados_H = []
fig, axes = plt.subplots(1, 5, figsize=(20, 3.8))

for ax, H in zip(axes, [1, 2, 4, 8, 64]):
    m_h, _ = entrenar(H=H)
    acc_h  = m_h.evaluate(X_test, y_test, verbose=0)[1]
    resultados_H.append((H, m_h.count_params(), acc_h))
    plot_decision_boundary(m_h, X_test, y_test, ax, f'H = {H}  (acc = {acc_h:.3f})')

plt.tight_layout(); plt.show()

print(f"{'H':>4} | {'parámetros':>10} | {'acc prueba':>10}")
print('-' * 32)
for H, npar, acc_h in resultados_H:
    print(f'{H:>4} | {npar:>10} | {acc_h:>10.4f}')

> **Ojo con la lectura de esta tabla.** El salto no es suave: con $H=1$ y $H=2$ la red se queda
> en una frontera casi recta, y a partir de $H=8$ resuelve el problema. El caso intermedio
> ($H=4$) es el interesante: tiene capacidad *suficiente en teoría*, pero el entrenamiento no
> siempre encuentra la solución. Es la diferencia entre **poder representar** una función y
> **lograr aprenderla** — el matiz que el teorema de aproximación universal no cubre. Vuelve
> sobre esto en la pregunta 4 del cierre.

---
### Experimento C — La tasa de aprendizaje $\alpha$

Es el primer hiperparámetro que hay que ajustar a mano, y el que más entrenamientos arruina.

**Predice**: ¿qué forma tendrá la curva de pérdida con $\alpha = 0.0001$? ¿Y con $\alpha = 5$?

In [ ]:
plt.figure(figsize=(8, 4.5))
print(f"{'lr':>8} | {'pérdida final':>13} | {'oscilación':>10} | {'acc prueba':>10} | diagnóstico")
print('-' * 84)

for lr in [0.0001, 0.001, 0.01, 0.5, 5.0]:
    m_lr, h_lr = entrenar(lr=lr, epochs=100)
    curva = h_lr.history['loss']
    perd  = curva[-1]
    # Cuánto salta la pérdida de una época a otra al final: mide la inestabilidad del paso.
    osc   = float(np.mean(np.abs(np.diff(curva[-30:]))))
    acc   = m_lr.evaluate(X_test, y_test, verbose=0)[1]
    plt.plot(curva, lw=1.8, label=f'lr = {lr}')

    if   perd > 0.65:  diag = 'ROTO: se estancó en log 2 (predice 0.5 para todo)'
    elif perd > 0.35:  diag = 'demasiado lento: no converge a tiempo'
    elif osc > 0.005:  diag = 'llega, pero la curva oscila: el paso es grande'
    elif perd > 0.15:  diag = 'converge, pero lento'
    else:              diag = 'bien'
    print(f'{lr:>8} | {perd:>13.4f} | {osc:>10.4f} | {acc:>10.4f} | {diag}')

plt.axhline(np.log(2), color='gray', ls=':', lw=1.5)
plt.xlabel('época'); plt.ylabel('pérdida'); plt.yscale('log')
plt.title('Efecto de la tasa de aprendizaje'); plt.legend(); plt.grid(alpha=0.2)
plt.tight_layout(); plt.show()

> Fíjate en la escala logarítmica del eje $y$: con $\alpha = 5$ la pérdida **empieza disparada**
> —los primeros pasos son tan grandes que empeoran el modelo— y luego se queda plana en $\log 2$.
> La red no se "rompe" con un error: simplemente deja de aprender y predice $0.5$ para todo, que
> es justamente el $50\%$ de accuracy.

---
### Experimento D — Sobreajuste y *early stopping*

Hasta ahora no hemos visto sobreajuste porque el problema es fácil y hay datos de sobra. Vamos a
provocarlo a propósito: **menos datos, más ruido y una red mucho más grande** (3 capas de 256
unidades, más de 130.000 parámetros para 330 ejemplos).

`EarlyStopping` es un *callback*: una función que Keras ejecuta al final de cada época. Este
vigila `val_loss` y, si no mejora durante `patience` épocas seguidas, detiene el entrenamiento y
—con `restore_best_weights=True`— **devuelve los pesos de la mejor época**, no los últimos.

**TODO 5.** Crea el callback con `monitor='val_loss'`, `patience=40` y `restore_best_weights=True`.

In [ ]:
# Dataset a propósito difícil: pocos ejemplos y mucho más ruido.
Xr, yr = make_moons(n_samples=500, noise=0.35, random_state=SEED)
yr = yr.astype('float32')
Xr_tr, Xr_tmp, yr_tr, yr_tmp = train_test_split(Xr, yr, test_size=0.34, stratify=yr, random_state=SEED)
Xr_val, Xr_te, yr_val, yr_te = train_test_split(Xr_tmp, yr_tmp, test_size=0.5, stratify=yr_tmp, random_state=SEED)
mu_r, sd_r = Xr_tr.mean(axis=0), Xr_tr.std(axis=0)
Xr_tr  = ((Xr_tr  - mu_r) / sd_r).astype('float32')
Xr_val = ((Xr_val - mu_r) / sd_r).astype('float32')
Xr_te  = ((Xr_te  - mu_r) / sd_r).astype('float32')
datos_ruidosos = (Xr_tr, yr_tr, Xr_val, yr_val)

print(f'Entrenamiento: {len(Xr_tr)} | Validación: {len(Xr_val)} | Prueba: {len(Xr_te)}')
print('Red grande: 3 capas ocultas de 256 unidades para 330 ejemplos de entrenamiento.')

In [ ]:
# TODO 5: crea el callback de early stopping.
early_stopping = ...

# Sin early stopping: dejamos correr las 300 épocas completas.
m_over, h_over = entrenar(H=256, capas_ocultas=3, lr=0.005, epochs=300,
                          batch_size=16, datos=datos_ruidosos)

# Con early stopping: el callback decide cuándo parar.
m_es, h_es = entrenar(H=256, capas_ocultas=3, lr=0.005, epochs=300,
                      batch_size=16, datos=datos_ruidosos, callbacks=[early_stopping])

print(f'Parámetros de la red: {m_over.count_params():,} para {len(Xr_tr)} ejemplos')
print(f'\nSIN early stopping: corrió las {len(h_over.history["loss"])} épocas')
print(f'CON early stopping: se detuvo en la época {len(h_es.history["loss"])}')

In [ ]:
# --- Validación TODO 5 ---
assert isinstance(early_stopping, keras.callbacks.EarlyStopping), \
    'early_stopping debe ser un keras.callbacks.EarlyStopping'
assert early_stopping.monitor == 'val_loss', \
    f"Debe vigilar 'val_loss', vigila '{early_stopping.monitor}'"
assert early_stopping.restore_best_weights, \
    'Sin restore_best_weights=True te quedas con los pesos del final, no con los mejores'
assert len(h_es.history['loss']) < len(h_over.history['loss']), \
    'El entrenamiento con early stopping debería haberse detenido antes'

ho = h_over.history
mejor = int(np.argmin(ho['val_loss']))

print('OK TODO 5 - early stopping configurado y funcionando.\n')
print(f"Sin early stopping, dejándolo correr las 300 épocas:")
print(f"  val_loss mínima : {min(ho['val_loss']):.4f} en la época {mejor}")
print(f"  val_loss final  : {ho['val_loss'][-1]:.4f}   <- MUCHO peor que el mínimo")
print(f"  accuracy entrenamiento final : {ho['accuracy'][-1]:.4f}")
print(f"  accuracy validación   final : {ho['val_accuracy'][-1]:.4f}   <- la brecha es el sobreajuste")
print(f"\nCon early stopping:")
print(f"  se detuvo en la época   : {len(h_es.history['loss'])}")
print(f"  pesos que se conservaron: los de la época {early_stopping.best_epoch}"
      f"  (val_loss = {early_stopping.best:.4f})")
print(f"  -> coincide con el mínimo de arriba: es exactamente el modelo que queríamos.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(ho['loss'], label='entrenamiento', lw=2)
axes[0].plot(ho['val_loss'], label='validación', lw=2, ls='--')
axes[0].axvline(mejor, color='red', ls=':', lw=2, label=f'mejor época ({mejor})')
axes[0].axvline(len(h_es.history['loss']) - 1, color='green', ls=':', lw=2,
                label=f'early stopping paró aquí ({len(h_es.history["loss"])-1})')
axes[0].set_xlabel('época'); axes[0].set_ylabel('pérdida')
axes[0].set_title('Pérdida: el sobreajuste se ve aquí'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.2)

axes[1].plot(ho['accuracy'], label='entrenamiento', lw=2)
axes[1].plot(ho['val_accuracy'], label='validación', lw=2, ls='--')
axes[1].set_xlabel('época'); axes[1].set_ylabel('accuracy')
axes[1].set_title('Accuracy: la brecha crece'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.2)
plt.tight_layout(); plt.show()

print(f"\nAccuracy en prueba sin early stopping : {m_over.evaluate(Xr_te, yr_te, verbose=0)[1]:.4f}")
print(f"Accuracy en prueba con early stopping : {m_es.evaluate(Xr_te, yr_te, verbose=0)[1]:.4f}")

> **Lee la gráfica de la izquierda con cuidado.** La pérdida de entrenamiento sigue bajando
> mientras la de validación toca su mínimo muy pronto y después **sube durante el resto del
> entrenamiento**. Todo lo que la red aprende a partir de ese punto es ruido del conjunto de
> entrenamiento: memorización, no generalización.
>
> `EarlyStopping` no hace al modelo más potente: hace que te quedes con el modelo **del punto
> correcto** en vez del último, y de paso ahorra dos tercios del tiempo de cómputo.
>
> Cuidado al comparar los dos accuracy en prueba: el conjunto tiene solo 85 ejemplos, así que una
> diferencia de dos puntos son **dos ejemplos** y no debe leerse como una mejora sólida. La
> evidencia contundente está en la `val_loss`, que mide la *confianza* de las predicciones y no
> solo si acierta: ahí la diferencia es de más del triple.
>
> **Un matiz sobre `patience`.** El callback es *codicioso*: para en cuanto pasan `patience`
> épocas sin mejorar, sin saber que más adelante podría mejorar. Con `patience` demasiado
> pequeña se detendría en un bache temporal y se quedaría con un modelo peor. Es el mismo dilema
> exploración/explotación que vimos en búsqueda local.

---
### Experimento E — Tu turno

**TODO 6.** Diseña y ejecuta **un** experimento propio. Algunas ideas (escoge una, o propón otra):

1. **Profundidad**: compara `capas_ocultas=1, 2, 4` manteniendo $H=8$. ¿Más capas es siempre mejor
   en un problema tan simple?
2. **Activación**: entrena con `activation='tanh'` y compara la frontera con la de ReLU.
   ¿Se nota que tanh es suave y ReLU es lineal a trozos?
3. **Semilla**: entrena cinco veces con `H=4` cambiando `SEED` dentro de `entrenar`. ¿Sale siempre
   el mismo accuracy? ¿Qué dice eso sobre los mínimos locales?
4. **`batch_size`**: prueba `batch_size` de 8, 32 y 256 con las mismas épocas. ¿Cuál converge en
   menos épocas? ¿Y en menos tiempo de reloj?
5. **Dropout**: inserta `keras.layers.Dropout(0.3)` entre las capas de la red grande del
   experimento D. ¿Reduce la brecha entre entrenamiento y validación?

Escribe tu conclusión en la celda de texto de abajo.

In [ ]:
# TODO 6: tu experimento aquí.

**Tu conclusión del experimento E:**

_(escribe aquí)_

---
## 9. Resumen de resultados

Completa la tabla con los números que obtuviste:

| Modelo | Parámetros | Forma de la frontera | Accuracy en prueba |
|---|---:|---|---:|
| Red 2→8→1 con ReLU | 33 | curva lineal a trozos | |
| Red 2→8→1 con `activation=None` | 33 | | |
| Red 2→1→1 ($H=1$) | 5 | | |
| Red 2→64→1 ($H=64$) | 257 | | |
| Red grande sin early stopping (datos ruidosos) | 132.609 | | |

---
## 10. Preguntas de cierre

Responde en las celdas de texto. Se evalúa el **razonamiento**, no la extensión.

### Sobre la arquitectura

1. En el experimento A, una red de 33 parámetros con `activation=None` resultó ser **exactamente**
   el mismo modelo que uno de 3 parámetros (la diferencia máxima fue del orden de $10^{-7}$).
   Explica algebraicamente por qué, y di qué pasaría si apiláramos 50 capas ocultas lineales.

2. La capa de salida de esta red es `Dense(1, activation='sigmoid')` con
   `loss='binary_crossentropy'`; en Fashion-MNIST era `Dense(10, activation='softmax')` con
   `loss='sparse_categorical_crossentropy'`. Explica qué cambia y qué se conserva entre los dos
   casos. ¿Por qué no serviría `Dense(1, activation='sigmoid')` para 10 clases?

3. `keras.Input(shape=(2,))` no aparece en `model.summary()` ni aporta parámetros. ¿Para qué
   sirve entonces? ¿Qué habría que cambiar en la red si el dataset tuviera 20 features en vez de 2?

### Sobre el entrenamiento

4. En el experimento B, $H=4$ dio peor accuracy que $H=8$, y también peor de lo que su capacidad
   permitiría. Como la red **puede representar** la solución con 4 neuronas, el problema no es de
   capacidad. ¿Qué otra cosa puede estar fallando? Relaciona tu respuesta con la inicialización
   aleatoria de los pesos y con la forma de la función de costo.

5. Con $\alpha = 5$ la pérdida se estancó exactamente en $\log 2 \approx 0.69$ y el accuracy en
   $0.5$. Explica por qué esos dos números aparecen juntos y qué le está pasando a los pesos.

6. `model.fit()` reemplaza en una línea el ciclo `forward → loss → backward → update`. Nombra
   **dos** cosas que ese ciclo hace por ti y que, si no las conocieras, te costaría depurar
   cuando el entrenamiento no funcione.

### Sobre la evaluación

7. Usamos tres conjuntos (entrenamiento, validación y prueba) en vez de dos. En el experimento D,
   `EarlyStopping` vigila `val_loss`. ¿Qué problema metodológico habría si lo hubiéramos
   configurado para vigilar la pérdida sobre el conjunto de **prueba**?

8. En el experimento D, la `val_loss` empieza a subir mientras la `val_accuracy` todavía se
   mantiene alta un rato más. ¿Cómo puede empeorar la pérdida sin que empeore (todavía) el
   accuracy? ¿Cuál de las dos avisa antes del sobreajuste?

9. Mira los ejemplos que la red falló en la sección 7 y sus probabilidades. ¿Son errores
   "graves"? ¿Qué te dice eso sobre el umbral de $0.5$, y en qué situación real convendría
   mover ese umbral hacia arriba o hacia abajo?

10. El teorema de aproximación universal garantiza que una red de dos capas con suficientes
    neuronas puede aproximar cualquier función continua. Sin embargo, pasar de $H=8$ a $H=64$
    casi no mejoró el accuracy en prueba. ¿Contradice esto al teorema? Justifica.

---

**Tus respuestas:**

_(escribe aquí)_

---
## 11. Para seguir explorando

- [playground.tensorflow.org](http://playground.tensorflow.org): el mismo experimento, interactivo.
  Prueba el dataset en espiral, quita la no linealidad, sube el *learning rate*.
- **Pasa a multiclase**: usa `make_blobs(n_samples=1200, centers=3)` y cambia solo la salida a
  `Dense(3, activation='softmax')` con `loss='sparse_categorical_crossentropy'`. El resto del
  notebook funciona igual — y esa es exactamente la red de `YourFirst_DNN-CNN_Keras.ipynb`.
- **Regularización L2**: añade `kernel_regularizer=keras.regularizers.l2(0.01)` a las capas de la
  red grande del experimento D y compara la brecha con la que obtuviste con early stopping.
- **Guarda y recarga tu modelo**: `model.save('red.keras')` y `keras.models.load_model('red.keras')`.
  Es lo que separa un experimento de un modelo que se puede usar.
- **El mismo modelo, sin Keras**: en `DNN_Class_exercise.ipynb` está esta misma red implementada
  a mano en NumPy, incluida la backpropagation que aquí hace `fit()` por ti.
- Próxima clase: **redes convolucionales (CNN)**, donde la arquitectura deja de ser
  *fully-connected* y empieza a aprovechar la estructura espacial de la entrada.